<a href="https://colab.research.google.com/github/zackdihel/ECON-5200-Data-Analytics/blob/main/Assignment%203/Econ_5200_Assignment_3_Causal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

In [3]:
#data generating
np.random.seed(42)
zeroes = np.zeros(100)
tips = np.random.exponential(scale=5.0, size=150)
driver_tips = np.concatenate([zeroes,tips])

In [8]:
#bootstrap engine
bootstrap_tips = np.array([
    np.median(np.random.choice(driver_tips, size=len(driver_tips), replace=True))
    for _ in range(10000)
])
bootstrap_tips

array([1.29902578, 0.48858193, 0.64458156, ..., 1.00822603, 1.00822603,
       0.32445088])

In [11]:
np.percentile(bootstrap_tips,2.5)

np.float64(0.2642554369019624)

In [12]:
np.percentile(bootstrap_tips,97.5)

np.float64(1.366359949751731)

In [23]:
median_tips_CI = np.percentile(bootstrap_tips, [2.5,97.5])
print(f"Confidence interval of median tips: [{median_tips_CI[0]:.2f}, {median_tips_CI[1]:.2f}]")

Confidence interval of median tips: [0.26, 1.37]


In [60]:
#Compared to untransformed data
tips_CI_normal = np.percentile(driver_tips, [2.5,97.5])
print(f"Confidence interval of observed tips: [{tips_CI_normal[0]:.2f}, {tips_CI_normal[1]:.2f}]")

Confidence interval of observed tips: [0.00, 15.01]


The confidence interval for median data shows a much smaller interval, less influenced by the skew we would see with data that was not transformed.

**Phase 2 - A/B test**

In [35]:
#control/treated
control = np.random.normal(loc=35,scale=5,size=500)
treated = np.random.lognormal(mean=3.4,sigma=0.4,size=500)

In [59]:
#difference in means
def difference_in_means(x,y):
  return np.mean(x) - np.mean(y)

diff = difference_in_means(control,treated)
print(f"Raw Effect of treatment: {diff:.2f} minutes saved")

Raw Effect of treatment: 2.87 minutes saved


In [58]:
loops = 5000
permutations_diff = np.zeros(loops)

for i in range(loops):
  deliveries = np.concatenate([control,treated])
  permutations = np.random.permutation(deliveries)
  control_perm = permutations[:500]
  treated_perm = permutations[500:]

  permutations_diff[i] = difference_in_means(control_perm,treated_perm)

p_val = np.mean(np.abs(permutations_diff) >= np.abs(diff))
print(f"P-Value of permutation: {p_val:.5f}")

P-Value of permutation: 0.00020


Only 0.02% of permutations would return a result as large as the observed 2.87. We can reject the null hypothesis that the control and treated groups are equal.

**Phase 3**

In [61]:
#load data
df = pd.read_csv("swiftcart_loyalty.csv")

In [62]:
df

,subscriber,pre_spend,account_age,support_tickets,post_spend
0,1,57.450712,37,2,85.169648
1,1,47.926035,41,0,72.802404
2,1,59.715328,41,0,79.858905
3,1,72.845448,34,0,80.335466
4,1,46.487699,34,2,67.956227
...,...,...,...,...,...
8936,1,35.172065,51,0,55.662507
8937,1,83.613898,5,2,94.767676
8938,0,57.146453,6,2,58.616370
8939,0,47.701092,13,0,60.069619


In [73]:
naive_diff = df[df.subscriber==1]['post_spend'].mean() - df[df.subscriber==0]['post_spend'].mean()
naive_diff

np.float64(17.57066938452379)

In [79]:
#covariates
X = df[['pre_spend','account_age','support_tickets','post_spend']]
Y = df['subscriber']

#propensity model
logit = LogisticRegression(solver='liblinear')
logit.fit(X,Y)

df['pscore'] = logit.predict_proba(X)[:,1]
df[['subscriber','pscore']].head()

,subscriber,pscore
0,1,0.994809
1,1,0.975515
2,1,0.965891
3,1,0.533667
4,1,0.837345


In [85]:
#NN
subscribed = df[df.subscriber==1]
unsubscribed = df[df.subscriber==0]

nbrs = NearestNeighbors(n_neighbors=1).fit(unsubscribed[['pscore']])

distances, indices = nbrs.kneighbors(subscribed[['pscore']])
matched = unsubscribed.iloc[indices.flatten()]

matched_df = pd.concat([subscribed,matched])

print(f"Original subscribers: {len(subscribed)}")
print(f"Matched subscribers: {len(matched)}")

Original subscribers: 4200
Matched subscribers: 4200


In [91]:
print(f"[Before] Raw Effect (Difference): ${naive_diff:,.2f}")

#causal effect for matched outcomes
matched_sub = matched_df[matched_df.subscriber==1]['post_spend']
matched_unsub = matched_df[matched_df.subscriber==0]['post_spend']

matched_diff = matched_sub.mean() - matched_unsub.mean()

print(f"[After] Matched difference: ${matched_diff:,.2f}")

[Before] Raw Effect (Difference): $17.57
[After] Matched difference: $1.04


The interpretation of this effect is that our initial data on spending was that subscribers spend $17.57 more than non-subscribers. However this is just a simple difference, and likely influenced by factors not captured in that math. After linking subscriber data to users with similar profiles, using the propensity model, we find that the truer difference in spending is a little over one dollar.